# 03 — Advanced Typing

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from typing import Generic, Sequence, TypeVar
T = TypeVar("T")

# Provided for this lesson.
from typing import Optional

from lesson import Exercises, ___, run

check = Exercises('advanced/03', {1: '37a27d8b23079106', 2: '601b3c705c5f7379', 3: '233b7c562d63eb92', 4: '2342b4f297348781', 5: '8f38827dcb9cda1c', 6: '8db8e732b6304ca3', 7: '610ae726d9ab8d3f'}, practice='advanced_01_internals_typing_architecture', independent='first_or_none')

## 1. TypeVar and generics

- `T = TypeVar("T")` links types: `def first(xs: Sequence[T]) -> T | None` returns the same type it received.
- A generic class: `class Stack(Generic[T])`, then `Stack[int]()` for type checkers. At runtime nothing is enforced.
- Python 3.12 also allows `def first[T](xs: list[T]) -> T` and `class Stack[T]:`.

In [ ]:
from typing import Generic, TypeVar, Sequence
T = TypeVar("T")

def first_or_none(values: Sequence[T]) -> T | None:
    return values[0] if values else None

class Stack(Generic[T]):
    def __init__(self):
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

s = Stack[int]()
s.push(1); s.push(2)
print(s.pop(), first_or_none("abc"), first_or_none([]))

**Exercise 1.** Return the first element or `None`.

In [ ]:
def first_or_none(values):
    return values[0] if values else ___

check(1, (first_or_none([7, 8]), first_or_none(())))

**Exercise 2.** A stack pops the last item pushed. Push 1, 2, 3, then pop twice.

In [ ]:
items = []
for x in [1, 2, 3]:
    items.append(x)
popped = [items.pop(), items.pop()]
predicted = ___
check(2, predicted)

**Exercise 3.** Raise the right error on an empty pop. Write its class name.

In [ ]:
answer = ___
check(3, answer)

**Exercise 4.** Is `Stack[int]` enforced at runtime? Push a string onto it and predict whether it's accepted.

In [ ]:
from typing import Generic, TypeVar
T = TypeVar("T")
class Stack(Generic[T]):
    def __init__(self): self.items = []
    def push(self, x: T): self.items.append(x)

s = Stack[int]()
s.push("oops")
predicted = ___   # True if accepted
check(4, predicted)

## 2. Runtime checks from annotations

- `inspect.signature(func)` describes the parameters; `.bind(*args, **kwargs)` maps call arguments to names.
- `typing.get_type_hints(func)` returns the annotations as real objects.
- Only check plain classes with `isinstance`; skip generics like `list[int]`.

In [ ]:
import inspect, typing

def greet(name: str, times: int = 1) -> str:
    return name * times

sig = inspect.signature(greet)
bound = sig.bind("hi", times=2)
print(dict(bound.arguments))
print(typing.get_type_hints(greet))

**Exercise 5.** Bind a call to parameter names.

In [ ]:
import inspect
def f(a, b, c=0): pass
bound = inspect.signature(f).bind(1, ___, c=3)
check(5, dict(bound.arguments))   # make b equal 2

**Exercise 6.** Check a value against its annotation.

In [ ]:
hints = {"age": int}
value = "41"
ok = isinstance(value, hints[___])
check(6, ok)

**Exercise 7.** Is `list[int]` a plain class you can pass to `isinstance`? Write `True` or `False`.

In [ ]:
predicted = ___   # isinstance(list[int], type)
check(7, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `Stack`

A typed LIFO stack; popping an empty one raises `IndexError`.

In [ ]:
class Stack(Generic[T]):
    """A typed LIFO stack. pop() on an empty stack raises IndexError('pop from empty stack')."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(___)

    def pop(self) -> T:
        if ___:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def __len__(self) -> int:
        return ___

In [ ]:
check('Stack')

### Independent build — `first_or_none`

Return the first item of a sequence, or None when empty. Preserve the element type in the public annotation so a caller passing a sequence of integers can infer int or None.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def first_or_none(values: Sequence[T]) -> Optional[T]:
    raise NotImplementedError

In [ ]:
check('first_or_none')

### 3. `enforce_types`

A decorator that raises `TypeError` when an argument doesn't match its annotation.

In [ ]:
def enforce_types(func):
    """Decorator: raise TypeError when an argument is not an instance of its annotation.

    Only check parameters annotated with a plain class (int, str, ...); skip others.
    Hint: inspect.signature(func).bind(*args, **kwargs).
    """
    import functools
    signature = ___
    hints = ___

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = signature.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = hints.get(name)
            if ___:
                raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")
        return ___
    return wrapper

In [ ]:
check('enforce_types')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A TypeVar in def first(xs: list[T]) -> T expresses…

   - **a)** T must be int
   - **b)** Any return type
   - **c)** The return type matches the element type

**2.** Protocol-based typing is…

   - **a)** Structural (has the right methods)
   - **b)** Nominal (inheritance-based)
   - **c)** Runtime-enforced

**3.** @overload is for…

   - **a)** Operator overloading
   - **b)** Documenting different return types for different argument types
   - **c)** Faster functions

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/03', {1: 'c76b720aff9a', 2: 'aff427a20652', 3: '9b1119499e51'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [04 — Imports, Introspection, and Plugin Architecture](04_imports_introspection_plugins.ipynb)